# oracle_hr_demo — Pipeline Walkthrough

Demonstrates ingesting from an Oracle-style HR database with a **bronze-layer SQL filter**,
a silver join across three tables, and three gold aggregations.

| Layer | What happens |
|-------|--------------|
| Bronze | dlt reads `employees`, `departments`, `jobs` from SQLite (or Oracle) |
| | `filter:` pushes `WHERE department_id IN (10,20,60,80,90)` to the source |
| | `select:` drops `email` (PII) before data enters the lake |
| Silver | Type-cast each table; UDF joins all three into `employees_enriched` |
| Gold | Three aggregations: headcount by dept · salary by job · salary utilisation |

**Data layout after a full run:**
```
oracle_hr/
├── data/
│   ├── source/
│   │   └── oracle_hr.db          ← SQLite source (setup_db.py)
│   ├── bronze/
│   │   ├── employees/            ← dlt raw shards
│   │   ├── departments/          ← dlt raw shards
│   │   ├── jobs/                 ← dlt raw shards
│   │   ├── employees.parquet     ← 12 rows (3 in Purchasing/Shipping excluded)
│   │   ├── departments.parquet   ← 7 rows
│   │   └── jobs.parquet          ← 8 rows
│   ├── silver/
│   │   ├── employees.parquet
│   │   ├── departments.parquet
│   │   ├── jobs.parquet
│   │   └── employees_enriched.parquet   ← 12 rows × joined columns
│   └── gold/oracle_hr/
│       ├── headcount_by_department.parquet
│       ├── salary_by_job.parquet
│       └── salary_utilization.parquet
```

Run cells top to bottom.

## Setup — navigate to example root

In [1]:
import os
import polars as pl
from pathlib import Path

# ipynb/ → oracle_hr/ → oracle_hr_demo/
example_root = Path(os.getcwd()).parent.parent
os.chdir(example_root)
print(f'Working directory: {os.getcwd()}')

Working directory: /home/ht/Documents/HT_GitHub/openmedallion/examples/oracle_hr_demo


## Seed — create the HR database

Creates `oracle_hr/data/source/oracle_hr.db` with:
- **15 employees** across 7 departments (12 pass the bronze filter — all in target depts, ACTIVE and INACTIVE; 3 in Purchasing/Shipping are excluded)
- **7 departments** (5 in scope: Admin, Marketing, IT, Sales, Executive)
- **8 jobs** with salary bands (used by the gold pre-agg UDF)

In [2]:
!python setup_db.py

✅  Database seeded at oracle_hr/data/source/oracle_hr.db

   15 employees inserted:
    • 12 in target departments  (10, 20, 60, 80, 90)  → ingested by bronze
    •  3 in dept 30 / 50        → excluded by department filter

   Bronze filter: department only.  All 12 employees (ACTIVE and INACTIVE) are ingested.

Next steps:
  medallion run oracle_hr --projects . --layer bronze
  medallion run oracle_hr --projects . --layer silver
  medallion run oracle_hr --projects .

To use a real Oracle/Postgres database:
  1. cp ../secrets.yaml.example ../secrets.yaml  and fill in your credentials
  2. Edit oracle_hr/backend/bronze.yaml — swap connection_string for credentials_file


## Inspect source data

In [3]:
import sqlite3

con = sqlite3.connect('oracle_hr/data/source/oracle_hr.db')

print('── employees (15 total in DB) ──')
print(pl.read_database('SELECT * FROM employees ORDER BY department_id, employee_id', con))

print('\n── departments ──')
print(pl.read_database('SELECT * FROM departments ORDER BY department_id', con))

print('\n── jobs ──')
print(pl.read_database('SELECT * FROM jobs ORDER BY job_id', con))

con.close()

── employees (15 total in DB) ──
shape: (15, 10)
┌────────────┬────────────┬───────────┬──────────┬───┬─────────┬────────────┬───────────┬──────────┐
│ employee_i ┆ first_name ┆ last_name ┆ email    ┆ … ┆ salary  ┆ manager_id ┆ departmen ┆ status   │
│ d          ┆ ---        ┆ ---       ┆ ---      ┆   ┆ ---     ┆ ---        ┆ t_id      ┆ ---      │
│ ---        ┆ str        ┆ str       ┆ str      ┆   ┆ f64     ┆ i64        ┆ ---       ┆ str      │
│ i64        ┆            ┆           ┆          ┆   ┆         ┆            ┆ i64       ┆          │
╞════════════╪════════════╪═══════════╪══════════╪═══╪═════════╪════════════╪═══════════╪══════════╡
│ 200        ┆ Jennifer   ┆ Whalen    ┆ JWHALEN  ┆ … ┆ 4400.0  ┆ 101        ┆ 10        ┆ ACTIVE   │
│ 201        ┆ Michael    ┆ Hartstein ┆ MHARTSTE ┆ … ┆ 13000.0 ┆ 100        ┆ 20        ┆ ACTIVE   │
│ 202        ┆ Pat        ┆ Fay       ┆ PFAY     ┆ … ┆ 6000.0  ┆ 201        ┆ 20        ┆ ACTIVE   │
│ 114        ┆ Den        ┆ Raphaely  ┆ DR

---
## Bronze — filtered ingestion

The `filter` field in `backend/bronze.yaml` applies:
```
WHERE department_id IN (10, 20, 60, 80, 90)
```
This is pushed directly to the SQL query — rows in dept 30 (Purchasing) and dept 50 (Shipping)
never enter the data lake. Both ACTIVE and INACTIVE employees in the target departments are ingested.

The `select` field drops `email` (PII) at source — it never appears in bronze or downstream.

In [4]:
!medallion run oracle_hr --layer bronze


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  medallion  ·  oracle_hr  ·  bronze ingestion
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📋  [config] bronze: oracle_hr/backend/bronze.yaml
📋  [config] silver: oracle_hr/backend/silver.yaml
📋  [config] gold  : oracle_hr/backend/gold.yaml

── Bronze ─────────────────────────────────────────────────
🔌  Connecting to sqlite at oracle_hr/data/source/oracle_hr.db (default schema) ...
✅  Connected — 3 table(s) in default schema:
    departments  employees    jobs
2026-05-14 19:27:05,046|[WARNING]|175782|140604860825920|dlt|filesystem.py|prepare_load_table:860|Falling back to `append` on `departments`.
2026-05-14 19:27:05,048|[WARNING]|175782|140604860825920|dlt|filesystem.py|prepare_load_table:860|Falling back to `append` on `jobs`.
2026-05-14 19:27:05,053|[WARNING]|175782|140604860825920|dlt|filesystem.py|prepare_load_table:860|Falling back to `append` on `departments`.
2026-05-14 19:27:05,055|[WARNING]|175782|1406

In [5]:
bronze_dir = Path('data/bronze')

print('── employees.parquet (bronze — after filter) ──')
emp_bronze = pl.read_parquet(bronze_dir / 'employees.parquet')
print(f'Shape: {emp_bronze.shape}  ← 12 rows (not 15; 3 in Purchasing/Shipping excluded)')
print(emp_bronze.select(['employee_id','first_name','last_name','department_id','hire_date','salary']))

print(f'\nUnique department_id values: {sorted(emp_bronze["department_id"].unique().to_list())}')

── employees.parquet (bronze — after filter) ──
Shape: (12, 8)  ← 12 rows (not 15; 3 in Purchasing/Shipping excluded)
shape: (12, 6)
┌─────────────┬────────────┬───────────┬───────────────┬────────────┬─────────┐
│ employee_id ┆ first_name ┆ last_name ┆ department_id ┆ hire_date  ┆ salary  │
│ ---         ┆ ---        ┆ ---       ┆ ---           ┆ ---        ┆ ---     │
│ i64         ┆ str        ┆ str       ┆ i64           ┆ str        ┆ f64     │
╞═════════════╪════════════╪═══════════╪═══════════════╪════════════╪═════════╡
│ 100         ┆ Steven     ┆ King      ┆ 90            ┆ 2003-06-17 ┆ 24000.0 │
│ 101         ┆ Neena      ┆ Kochhar   ┆ 90            ┆ 2005-09-21 ┆ 17000.0 │
│ 102         ┆ Lex        ┆ De Haan   ┆ 90            ┆ 2001-01-13 ┆ 17000.0 │
│ 103         ┆ Alexander  ┆ Hunold    ┆ 60            ┆ 2006-01-03 ┆ 9000.0  │
│ 104         ┆ Bruce      ┆ Ernst     ┆ 60            ┆ 2007-05-21 ┆ 6000.0  │
│ …           ┆ …          ┆ …         ┆ …             ┆ …         

In [6]:
print('── departments.parquet (all 7 rows — no filter on this table) ──')
print(pl.read_parquet(bronze_dir / 'departments.parquet')
        .select(['department_id','department_name']))

print('\n── jobs.parquet (all 8 rows) ──')
print(pl.read_parquet(bronze_dir / 'jobs.parquet')
        .select(['job_id','job_title','min_salary','max_salary']))

── departments.parquet (all 7 rows — no filter on this table) ──
shape: (14, 2)
┌───────────────┬─────────────────┐
│ department_id ┆ department_name │
│ ---           ┆ ---             │
│ i64           ┆ str             │
╞═══════════════╪═════════════════╡
│ 10            ┆ Administration  │
│ 20            ┆ Marketing       │
│ 30            ┆ Purchasing      │
│ 50            ┆ Shipping        │
│ 60            ┆ IT              │
│ …             ┆ …               │
│ 30            ┆ Purchasing      │
│ 50            ┆ Shipping        │
│ 60            ┆ IT              │
│ 80            ┆ Sales           │
│ 90            ┆ Executive       │
└───────────────┴─────────────────┘

── jobs.parquet (all 8 rows) ──
shape: (16, 4)
┌─────────┬───────────────────────────────┬────────────┬────────────┐
│ job_id  ┆ job_title                     ┆ min_salary ┆ max_salary │
│ ---     ┆ ---                           ┆ ---        ┆ ---        │
│ str     ┆ str                           ┆ f64   

---
## Live Dashboard

The `PipelineDashboard` widget polls `pipeline_status.json` every 500 ms and renders
inline in Jupyter. Panel ships as a core dependency — no extra install needed.

**With `--track`** — run the pipeline in a terminal using
`medallion run oracle_hr --projects . --track`. The dashboard updates live as each
Bronze → Silver → Gold node completes.

**Without `--track`** — the widget shows IDLE with a hint. No errors thrown.

**Using Kestra instead?** Use `make kestra-up` and the Kestra UI at `http://localhost:8080`
for scheduled / production runs. `PipelineDashboard` is for interactive notebook dev only.

In [7]:
!uv sync --extra notebook

Resolved 141 packages in 4ms
   Building openmedallion @ file:///home/ht/Documents/HT_GitHub/openmedallion
   Building openmedallion @ file:///home/ht/Documents/HT_GitHub/openmedallion
   Building openmedallion @ file:///home/ht/Documents/HT_GitHub/openmedallion
   Building openmedallion @ file:///home/ht/Documents/HT_GitHub/openmedallion
   Building openmedallion @ file:///home/ht/Documents/HT_GitHub/openmedallion
   Building openmedallion @ file:///home/ht/Documents/HT_GitHub/openmedallion
   Building openmedallion @ file:///home/ht/Documents/HT_GitHub/openmedallion
      Built openmedallion @ file:///home/ht/Documents/HT_GitHub/openmedallion
Prepared 1 package in 1.36s                                              
Uninstalled 1 package in 1ms
Installed 1 package in 3ms.5.4 (from file:///home/ht/Documen
 ~ openmedallion==2026.5.4 (from file:///home/ht/Documents/HT_GitHub/openmedallion)


In [6]:
from openmedallion.viz.notebook import PipelineDashboard
dash = PipelineDashboard()   # reads pipeline_status.json in CWD
dash.show()                  # renders inline; updates every 500 ms
# For live progress, run in a terminal:
#   medallion run oracle_hr --projects . --track

BokehModel(combine_events=True, render_bundle={'docs_json': {'9639764a-e789-47df-8c61-f046283a5248': {'version…

---
## Silver — type-cast + enrichment join

Phase 1: cast columns to correct types.  
Phase 2: UDF (`build_employees_enriched`) joins employees → departments → jobs into one wide table.

In [7]:
!medallion run oracle_hr --projects . --layer silver


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  medallion  ·  oracle_hr  ·  bronze → silver
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📋  [config] bronze: oracle_hr/backend/bronze.yaml
📋  [config] silver: oracle_hr/backend/silver.yaml
📋  [config] gold  : oracle_hr/backend/gold.yaml

  ⏭️  bronze  skipped (existing files)

── Silver ─────────────────────────────────────────────────
🔧  [silver] base    employees.parquet → employees.parquet  (12 rows)
🔧  [silver] base    departments.parquet → departments.parquet  (14 rows)
🔧  [silver] base    jobs.parquet → jobs.parquet  (16 rows)
🔧  [silver] derived employees_enriched.parquet  (48 rows)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✅  bronze → silver complete.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



In [8]:
silver_dir = Path('data/silver')

print('── employees_enriched.parquet (derived table) ──')
enriched = pl.read_parquet(silver_dir / 'employees_enriched.parquet')
print(f'Shape: {enriched.shape}')
print(enriched.select([
    'first_name', 'last_name', 'salary',
    'department_name', 'job_title',
    'min_salary', 'max_salary'
]).sort('department_name', 'salary', descending=[False, True]))

── employees_enriched.parquet (derived table) ──
Shape: (48, 12)
shape: (48, 7)
┌────────────┬───────────┬─────────┬─────────────────┬────────────────┬────────────┬────────────┐
│ first_name ┆ last_name ┆ salary  ┆ department_name ┆ job_title      ┆ min_salary ┆ max_salary │
│ ---        ┆ ---       ┆ ---     ┆ ---             ┆ ---            ┆ ---        ┆ ---        │
│ str        ┆ str       ┆ f64     ┆ str             ┆ str            ┆ f64        ┆ f64        │
╞════════════╪═══════════╪═════════╪═════════════════╪════════════════╪════════════╪════════════╡
│ Jennifer   ┆ Whalen    ┆ 4400.0  ┆ Administration  ┆ Administration ┆ 3000.0     ┆ 6000.0     │
│            ┆           ┆         ┆                 ┆ Assistant      ┆            ┆            │
│ Jennifer   ┆ Whalen    ┆ 4400.0  ┆ Administration  ┆ Administration ┆ 3000.0     ┆ 6000.0     │
│            ┆           ┆         ┆                 ┆ Assistant      ┆            ┆            │
│ Jennifer   ┆ Whalen    ┆ 4400.0  ┆ A

---
## Gold — three aggregations

1. **headcount_by_department** — headcount, total payroll, avg salary per department  
2. **salary_by_job** — min/max/avg actual salary per job title  
3. **salary_utilization** — avg salary as % of job-band max, per dept × job (pre-agg UDF adds `salary_pct_of_max`)

In [9]:
!medallion run oracle_hr --projects .


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  medallion  ·  oracle_hr  ·  bronze → silver → gold
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📋  [config] bronze: oracle_hr/backend/bronze.yaml
📋  [config] silver: oracle_hr/backend/silver.yaml
📋  [config] gold  : oracle_hr/backend/gold.yaml

  ⏭️  bronze  skipped (existing files)
  ⏭️  silver  skipped (existing files)

── Gold ───────────────────────────────────────────────────
📊  [gold/oracle_hr] headcount_by_department.parquet  (5 rows)
📊  [gold/oracle_hr] salary_by_job.parquet  (8 rows)
⚙️   [gold]  udf add_salary_metrics()  48 → 48 rows
📊  [gold/oracle_hr] salary_utilization.parquet  (8 rows)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✅  bronze → silver → gold complete.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



In [10]:
gold_dir = Path('data/gold/oracle_hr')

print('── headcount_by_department.parquet ──')
hc = pl.read_parquet(gold_dir / 'headcount_by_department.parquet')
print(hc.sort('total_payroll', descending=True))

print(f'\nTotal headcount across departments: {hc["headcount"].sum()}')
print(f'Total payroll: ${hc["total_payroll"].sum():,.0f}')

── headcount_by_department.parquet ──
shape: (5, 4)
┌─────────────────┬───────────┬───────────────┬──────────────┐
│ department_name ┆ headcount ┆ total_payroll ┆ avg_salary   │
│ ---             ┆ ---       ┆ ---           ┆ ---          │
│ str             ┆ u32       ┆ f64           ┆ f64          │
╞═════════════════╪═══════════╪═══════════════╪══════════════╡
│ Executive       ┆ 12        ┆ 232000.0      ┆ 19333.333333 │
│ Sales           ┆ 16        ┆ 202000.0      ┆ 12625.0      │
│ Marketing       ┆ 8         ┆ 76000.0       ┆ 9500.0       │
│ IT              ┆ 8         ┆ 60000.0       ┆ 7500.0       │
│ Administration  ┆ 4         ┆ 17600.0       ┆ 4400.0       │
└─────────────────┴───────────┴───────────────┴──────────────┘

Total headcount across departments: 48
Total payroll: $587,600


In [11]:
print('── salary_by_job.parquet ──')
print(pl.read_parquet(gold_dir / 'salary_by_job.parquet')
        .sort('avg_actual_salary', descending=True))

── salary_by_job.parquet ──
shape: (8, 5)
┌─────────────────────┬────────────────┬───────────────────┬───────────────────┬───────────────────┐
│ job_title           ┆ employee_count ┆ min_actual_salary ┆ max_actual_salary ┆ avg_actual_salary │
│ ---                 ┆ ---            ┆ ---               ┆ ---               ┆ ---               │
│ str                 ┆ u32            ┆ f64               ┆ f64               ┆ f64               │
╞═════════════════════╪════════════════╪═══════════════════╪═══════════════════╪═══════════════════╡
│ President           ┆ 4              ┆ 24000.0           ┆ 24000.0           ┆ 24000.0           │
│ Administration Vice ┆ 8              ┆ 17000.0           ┆ 17000.0           ┆ 17000.0           │
│ President           ┆                ┆                   ┆                   ┆                   │
│ Marketing Manager   ┆ 4              ┆ 13000.0           ┆ 13000.0           ┆ 13000.0           │
│ Sales               ┆ 8              ┆ 12000.0 

In [12]:
print('── salary_utilization.parquet ──')
print('(avg salary as % of job-band maximum, grouped by dept × job)')
util = pl.read_parquet(gold_dir / 'salary_utilization.parquet')
print(util.sort('avg_salary_pct_of_max', descending=True))

── salary_utilization.parquet ──
(avg salary as % of job-band maximum, grouped by dept × job)
shape: (8, 4)
┌─────────────────┬───────────────────────────────┬───────────────────────┬───────────┐
│ department_name ┆ job_title                     ┆ avg_salary_pct_of_max ┆ headcount │
│ ---             ┆ ---                           ┆ ---                   ┆ ---       │
│ str             ┆ str                           ┆ f64                   ┆ u32       │
╞═════════════════╪═══════════════════════════════╪═══════════════════════╪═══════════╡
│ Sales           ┆ Sales Representative          ┆ 100.0                 ┆ 8         │
│ Marketing       ┆ Marketing Manager             ┆ 66.666667             ┆ 4         │
│ IT              ┆ Programmer                    ┆ 58.333333             ┆ 8         │
│ Administration  ┆ Administration Assistant      ┆ 46.666667             ┆ 4         │
│ Marketing       ┆ Marketing Representative      ┆ 40.0                  ┆ 4         │
│ Sales     

---
## Connecting to a real Oracle / Postgres database

1. Edit `examples/secrets.yaml` (one level above this demo) — fill in your `oracle:` or `postgres:` block.  
   Copy from `examples/secrets.yaml.example` if it doesn't exist yet.
2. In `oracle_hr/backend/bronze.yaml`, comment out `connection_string` and uncomment `credentials_file: ../secrets.yaml` + `dialect: oracle` (or `postgres`, `mysql`, `mssql`).
3. Install the driver if needed:

```bash
pip install "openmedallion[oracle]"   # for Oracle
```

4. Re-run bronze:

```bash
medallion run oracle_hr --projects . --layer bronze
```

The `filter:` in `bronze.yaml` passes straight through to the Oracle query — no code changes needed.

## Things to Try

- **Change the department filter**: edit `backend/bronze.yaml` → `filter:` field, re-run bronze
- **Add a new employee**: insert a row into SQLite, then `rm -rf oracle_hr/data/bronze/` and re-run bronze
- **Add a new gold aggregation**: add a YAML block to `backend/gold.yaml` (e.g., avg salary by department and job)
- **Use a real Oracle DB**: follow the "Connecting to a real Oracle / Postgres database" section above